In [ ]:
#Tokenizer - BPE Algorithm
import re
from collections import defaultdict
from transformers import AutoTokenizer #will just use this for pretokenization: splitting each word in our corpus
import torch
from torch import nn
import numpy as np

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

class BPETokenizer:
  def __init__(self, vocab_size, pretokenizer=AutoTokenizer.from_pretrained("gpt2")):
    self.vocab_size = vocab_size
    self.merges = {}
    self.word_freqs = defaultdict(int)
    self.splits = {}
    self.vocab = []
    self.pretokenizer = pretokenizer


  def train(self, corpus):
    self.word_freqs = defaultdict(int)
    #pre tokenize
    print("Pretokenizing...")
    for text in corpus:
        words_with_offsets = self.pretokenizer.backend_tokenizer.pre_tokenizer.pre_tokenize_str(text)
        new_words = [word for word, offset in words_with_offsets]
        for word in new_words:
            self.word_freqs[word] += 1

    #create vocab
    print("Creating vocab...")
    alphabet = []
    for word in self.word_freqs.keys():
        for letter in word:
            if letter not in alphabet:
              alphabet.append(letter)
        alphabet.sort()
    self.vocab = ["<|endoftext>", "[UNK]"] + alphabet.copy() # Added [UNK] to initial vocab

    self.splits = {word: [c for c in word] for word in self.word_freqs.keys()}

    def compute_pair_freqs():
      pair_freqs = defaultdict(int)
      for word, freq in self.word_freqs.items():
        split = self.splits[word]
        for i in range(len(split) - 1):
          pair = (split[i], split[i+1])
          pair_freqs[pair] += freq
      return pair_freqs

    def get_max_pair(pair_freqs):
      best_pair = ""
      max_freq = None
      for pair, freq in pair_freqs.items():
        if max_freq is None or max_freq < freq:
          best_pair = pair
          max_freq = freq
      return best_pair, max_freq

    def merge_pair(a, b):
        for word in self.word_freqs:
            split = self.splits[word]
            if len(split) == 1:
                continue

            i = 0
            while i < len(split) - 1:
                if split[i] == a and split[i + 1] == b:
                    split = split[:i] + [a + b] + split[i + 2 :]
                else:
                    i += 1
            self.splits[word] = split

    print("Training...")
    while len(self.vocab) < self.vocab_size:
      pair_freqs = compute_pair_freqs()
      best_pair, max_freq = get_max_pair(pair_freqs)
      if max_freq is None or max_freq == 0: # Handle cases where no more merges can be made (max_freq is None means pair_freqs was empty)
          break
      merge_pair(*best_pair)
      self.merges[best_pair] = best_pair[0] + best_pair[1]
      self.vocab.append(best_pair[0] + best_pair[1])

  def encode(self, text):
    # Pre-tokenize the input text into words
    words_with_offsets = self.pretokenizer.backend_tokenizer.pre_tokenizer.pre_tokenize_str(text)
    pre_tokenized_words = [word for word, offset in words_with_offsets]

    encoded_tokens = []
    for word in pre_tokenized_words:
        # Start with individual characters for each word
        current_split = [c for c in word]

        # Apply learned merges greedily
        while True:
            best_merge_idx = -1
            best_merge_pair = None
            for i in range(len(current_split) - 1):
                pair = (current_split[i], current_split[i+1])
                if pair in self.merges:
                    # Find the earliest merge in the sequence to apply
                    if best_merge_idx == -1 or self.vocab.index(self.merges[pair]) < self.vocab.index(self.merges[best_merge_pair]):
                         best_merge_idx = i
                         best_merge_pair = pair

            if best_merge_idx != -1:
                merged_token = self.merges[best_merge_pair]
                current_split = current_split[:best_merge_idx] + [merged_token] + current_split[best_merge_idx + 2:]
            else:
                break # No more merges possible for this word

        # Map the final split tokens to their IDs from the vocabulary
        for token in current_split:
            if token in self.vocab:
                encoded_tokens.append(self.vocab.index(token))
            else:
                # Use [UNK] token for out-of-vocabulary (OOV) words
                if "[UNK]" not in self.vocab:
                    # Fallback: if somehow [UNK] isn't in vocab, add it.
                    self.vocab.append("[UNK]")
                encoded_tokens.append(self.vocab.index("[UNK]"))

    return encoded_tokens

  def decode(self, token_ids):
    decoded_text = []
    for token_id in token_ids:
      if 0 <= token_id < len(self.vocab):
        decoded_text.append(self.vocab[token_id])
      else:
        # Handle unknown token IDs, e.g., by skipping or using a placeholder
        decoded_text.append("[UNK]") # Placeholder for unknown IDs
    return "".join(decoded_text).replace("Ġ", " ").strip()

cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [ ]:

from torch.utils.data import Dataset, DataLoader
from datasets import load_dataset
raw_dataset = load_dataset("wikitext", "wikitext-2-raw-v1")
train_corpus = raw_dataset["train"]["text"]
val_corpus = raw_dataset["validation"]["text"]

class TextDataset(Dataset):
    def __init__(self, tokens, max_seq_len):
        self.max_seq_len = max_seq_len
        # pre-chunk all tokens into input/target pairs
        self.examples = []
        for i in range(0, len(tokens) - max_seq_len, max_seq_len):
            input_chunk  = tokens[i : i + max_seq_len]
            target_chunk = tokens[i+1 : i + max_seq_len + 1]
            self.examples.append((
                torch.tensor(input_chunk,  dtype=torch.long),
                torch.tensor(target_chunk, dtype=torch.long)
            ))

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        return self.examples[idx]

In [ ]:

tokenizer = BPETokenizer(vocab_size=4000) # You can adjust vocab_size
tokenizer.train(train_corpus)
print(tokenizer.tokenize("the cat sat on the mat"))




README.md: 0.00B [00:00, ?B/s]

wikitext-2-raw-v1/test-00000-of-00001.pa(…):   0%|          | 0.00/733k [00:00<?, ?B/s]

wikitext-2-raw-v1/train-00000-of-00001.p(…):   0%|          | 0.00/6.36M [00:00<?, ?B/s]

wikitext-2-raw-v1/validation-00000-of-00(…):   0%|          | 0.00/657k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/4358 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/36718 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3760 [00:00<?, ? examples/s]

Pretokenizing...
Creating vocab...
Training...
[85, 196, 1873, 3166, 264, 199, 1086]


In [ ]:
class GPTConfig:
    d_model     = 512
    n_heads     = 8
    d_head      = 64
    n_layers    = 6
    max_seq_len = 128

class TrainingConfig:
    batch_size  = 64
    lr          = 1e-4
    n_epochs    = 50

In [ ]:
class AttentionHead(nn.Module):
  def __init__(self, config):
    super().__init__()
    #since our q, k, v vectors are just basically different versions of our x vectors storing different info about the tokens,
    #we just make the weights that get applied to them nxn where n is our embedding dimension. Lets also say m is our d_head, the dimension
    #of each attention head. For single head attention, m = n, but in multiheaded attention we essentially split the information encoded
    #in each q, k, v across multiple heads. We can evenly distribute the dimension too and say m = n/num_heads.
    #Since x can be thought of as a matrix of seq_len*n dimension, where each vector x_i is an embedding for the ith token in our sequence,
    #we use the learned weights of n*m to get new vectors q, k, and v of seq_len*m, where qi, ki, and vi are the m*1 attention head
    #space q/k/v vectors of the ith token and just represent different information about these vectors.
    #The whole idea of learning our q k v weights is to better encode information in them. And remember, if we just have one attention head,
    #m=n, so our q k v weights are n*n, and our q k v matrices are seq_len*n with n*1 qi ki vi individual vectors

    self.dk = config.d_model // config.n_heads #m in the explanation above, we'll also need it as a scaling factor for attention calculation
    #to ensure values aren't too large

    self.wq = nn.Linear(config.d_model, self.dk, bias=False) #Just a cleaner way of doing W = torch.randn(d_model, d_head)
    #which also lets us call w(x) which outputs output = x*W.T + b where b is an optional bias. Most modern implementations don't use
    #a bias. GPT 2 specifically does, but for this we'll just turn it off.
    self.wk = nn.Linear(config.d_model, self.dk, bias=False)
    self.wv = nn.Linear(config.d_model, self.dk, bias=False)

    self.register_buffer('mask', torch.tril(torch.ones(config.max_seq_len, config.max_seq_len))) #mask init, will set to 0's


  def forward(self, x):
    B, T, C = x.shape # Batch size, Sequence length, Embedding dimension
    q = self.wq(x) #sequence (query) vectors - looking at tokens in sequence
    k = self.wk(x) #context (key) vectors - how much our sequence tokens look at every other token
    v = self.wv(x) #value vectors - what parts of our words semantically do we need to emphasize, considering mixing with attention

    #compute attention
    pre_scores = q @ k.transpose(-2, -1) / self.dk**0.5
    #@ operator is dot product. This is our raw attention score matrix, which is how much a token qi from our sequence vector
    #attends to token kj from our context vector
    #this is one way of thinking about q and k: they are both matrices for each token in our sequence representing some amount of semantic information in a vector for each word,
    #but we can think of q as containing vectors for each token we're looking at, and k as vectors for each token as context to the q vectors
    #when training, we pass batches of sequences at a time, so each of our input x is actually going to have dimension batch_size*seq_len*d_model
    #so, we do transpose (-2,-1) to essentially just swap the last two dimensions, which will always be our seq_len and d_model (or d_head for q k v)
    #Therefore, raw_attn_i = vector for token qi, and raw_attn_ij = how much token qi attends to token kj
    #dk, our head dimension, is just a scaling factor

    pre_scores = pre_scores.masked_fill(self.mask[:T, :T] == 0, float('-inf'))
    #here we mask all the future tokens in the sequence that have 0 attention with '-inf' so that when we next take the softmax, our raw scores
    #actually give 0 for future tokens. This makes sense because a given token in a sequence should only depend on/attend to previous tokens

    raw_scores = nn.functional.softmax(pre_scores, dim=-1) #now we softmax to get the actual raw scores, which are our raw attention weights as a probability distribution
    #for each vector a_i in our attention weight matrix, the attentions scores should add up to 1, representing the distribution of
    #attention token i gives to each preceeding token j.

    #now that we know how much each token qi attends to tokens kj, and have some raw weights to assign that, we now can store
    #the actual token vector's information that we should be paying attention to. We will store this in what we call our value, or v vectors.
    #Similarly to q and k, v is also just another way of storing information for each token in our sequence. The difference is that this time
    #we know how much each token depends on others as indicated by our raw scores

    #when we computed the prescores, we took our q matrix (seq_len*d_k) and multiplied it by the transpose of our k matrix (d_k*seq_len).
    #After masking out future tokens and softmaxing to get a probability distribution, this left us with an attention matrix of
    #seq_len*seq_len, which makes sense because each a_i is an attention vector for each token i, and each a_ij represents how much that token
    #depends on token j. Now that we have these attention weights in our vocabulary/sequence space, we need to transform back into our embedding space.
    #This is because we need to actually "embed" the attentions that we learned for each of our seq_len tokens into the original embedding space for our
    #attention heads, d_k, so that these vectors actually represent the semantic information of each token.
    #So, generally speaking, when we have vectors in our embedding dimension d_model or our attention head's proportion of it, d_k,
    #it means we are looking at the semantics of the word as reflected in our embedding model, which captures token relationships in terms
    #of distance in a space. On the other hand, when we are looking at our vectors in terms of the vocabulary space or our sequence length,
    #it means we are looking at specifically relationships between tokens in the context of our sequence. This is the part that lets us
    #capture dependencies.
    self.dropout = nn.Dropout(0.1)#prevent memorizing train set
    raw_scores = self.dropout(nn.functional.softmax(pre_scores, dim=-1))

    attention_out = raw_scores @ v

    #when we get our final attention scores by multiplying our raw scores by our value vectors, we go back from the
    #vocabulary/sequence length space to our embedding dimension/attention head dimension (which is a proportion of our total embedding space).
    #So, two tokens who are normally not super similar in our embedding space based on the raw vectors from our embedding model
    #effectively become more similar after going through attention. As in two far away tokens in our embedding space become closer
    #after being transformed in attention and learning that they actually depend on each other in a sequence.
    #Basically the output attention matrix (seq_len*d_k) which contains vectors (d_k*1) for each token gives us
    #vectors for each token that remap it in almost a new version of our original embedding space
    #based on their dependencies on other words.
    #So, words that are dependent on each other become closer in our new "value" space which, represents semantic similarity, and
    #focuses on important learned semantic information that we actually need to convey
    #Remember, the attention output is really just a weighted sum on our value vector, ao attention_out_i = w_i0*v_0+w_i1*v_1+...
    #Where w is our raw_scores. So two words with similar raw scores are going to approach the same final attention vectors in attention_out.
    #And in this case similar raw scores means that the two tokens attend similarly to each other token in the sequence or strongly to each other
    #A good example is the sentence "the animal didn't cross the street because it was too tired".
    #The word it needs to resolve to animal, because "it" refers to "animal".
    #After attention, the representation of it gets heavily mixed with the representation of animal, pulling them close together in
    #the space even though their raw embeddings were probably quite different.
    #One important nuance though is that this new space that our final attention_out lives in, while it is a seq_len*d_k (or d_model for single head)
    #dimension matrix, the actual positions in this space don't map directly with our original embedding model. Rather, this new space,
    #which we call the value space, is a projection of our original embedding space via w_v, our value weight matrix, which learns to control
    #what information is important to extract from our attention matrix and random initialized values. So, when we mutliply x and w_v to get
    #v our value vectors, that w_v throughout training learns what parts to weigh more heavily from x so that after attention we are correctly
    #capturing the information we need to feed forward. So, when we have a trained w_v, we are able to select semantic meaning from our words
    #dependent on context capture by attention, which depends on our ability to train w_k and w_q to produce good q and k vectors that effectively
    #capture how much words in the sequence depend on each other. Eventaully, for multiheaded attention, we need to go back from d_k to d_model,
    #and we do a concat later to do that.

    return attention_out




class MultiHeadAttention(nn.Module):
  def __init__(self, config):
    super().__init__()
    self.heads = nn.ModuleList([AttentionHead(config) for _ in range(config.n_heads)])
    self.proj = nn.Linear(config.d_model, config.d_model) #parameters: W_O. Our output attention weight that
    #learns how to mix the outputs of each attention head. this is also a learned weight. We could technically not use it, but then our model
    #would be a lot less expressive. This helps combine the outputs of each attention head better. By placing a parameter here, we know
    #that any loss gradients that hit this weight will be directly in the process of combining attention heads, so it just optimizes on the
    #combination step.

  def forward(self, x):
    head_outputs = [head.forward(x) for head in self.heads]
    concat_out = torch.cat(head_outputs, dim=-1)
    #since we have batch_size * seq_len * d_k (d_head) outputs from each attention head, we want to concatenate outputs along the d_k axis
    #Since d_k = d_model/num_heads, we can just append all the values from each attention head output to the corresponding batch_size*seq_len along the axis d_k.
    #that way, we get d_k*num_heads entries per token, giving us a batch_size*seq_len*d_model final tensor with all of the information from each attention head
    #concatenated together.
    return self.proj(concat_out) #we are already in the correct dimensionality space before projecting, so we could mathematically
    #just return our concatenated output of attention. However, this would just be attention head 1, followed by head 2, head 3, and so on
    #for each token, which doesn't mix what each head learned at all. The idea of this projection layer W_O is that we can learn how to
    #combine each of our attention head outputs.

class FeedForward(nn.Module):
  def __init__(self, config):
    super().__init__()
    self.net = nn.Sequential(
        nn.Linear(config.d_model, 4*config.d_model),
        nn.ReLU(),
        nn.Dropout(0.1),
        nn.Linear(4*config.d_model, config.d_model),
        nn.Dropout(0.1)
    )
    #ffn in a transformer generally follows a 2 (input, hidden, output) layer narrow -> wide -> narrow structure. We don't change the dimension of our vector in terms of
    #input to output, but expanding in the middle lets us discover some more complex patterns, and then in the end go into our original space

  def forward(self, x):
    return self.net(x) #just pass x, which will just be original sequence x + our output tensor
    #from multihead self attention, through the neural net

class TransformerBlock(nn.Module):
  def __init__(self, config):
    super().__init__()
    self.norm1 = nn.LayerNorm(config.d_model)
    self.attn = MultiHeadAttention(config)
    self.norm2 = nn.LayerNorm(config.d_model)
    self.ffn = FeedForward(config)

  def forward(self, x): #transformer block pass through. All the things we compute during attention and ffn which are basically transformations
  #and operations applied to our original x to learn some information are just added to our original sequence
    x = x + self.attn(self.norm1(x)) #calling the model like this auto calls the forward function
    x = x + self.ffn(self.norm2(x))
    return x

In [ ]:

'''
Note:
Embedding Layer
Really just serves as a lookup table where given a token ID, we can retrieve the corresponding embedding vector.
The actual embedding vectors are learned during training of the whole transformer, its not something we train indepenently.
At this stage, we just have token ids which are unique per each word and don't consider position.
Positional encoding happens right after we get the embedding vector: given a base embedding vector for a unique token id,
we can then add positional information before we enter the transformer block.
Similarly, positional encodings are also learned.
'''

class ToyGPT(nn.Module):
  def __init__(self, config):
    super().__init__()
    self.config = config
    self.token_emb = nn.Embedding(config.vocab_size, config.d_model)
    self.pos_emb = nn.Embedding(config.max_seq_len, config.d_model)
    self.blocks = nn.ModuleList([TransformerBlock(config) for _ in range(config.n_layers)])
    self.norm = nn.LayerNorm(config.d_model)
    self.head = nn.Linear(config.d_model, config.vocab_size, bias=False)



  def forward(self, token_ids):
    #first we get our token's vectors from the embedding model
    #(just a map from id to vector), and add the corresponding positional vectors
    #via element wise addition
    seq_len = token_ids.shape[1]
    positions = torch.arange(seq_len, device=token_ids.device)

    x = self.token_emb(token_ids) + self.pos_emb(positions)
    #x is our input sequence as embedding vectors, added with position vectors
    #we must now look through our sequence and determine how much each token
    #attends to one another, i.e pass to our attention mechanism

    #pass through transformer blocks: normalization->attention->normalization->ffn
    for block in self.blocks:
      x = block(x)
    #at this point, we have passed our input sequence through several transformer blocks,
    #iteratively refining on our original embedding vector going into our value space to have better encoded information
    #of all of the tokens in our sequence contextually, including position, the base embeddings, context via attention, non linearity and
    #expanded features via our ffn, and the benefits of separating our embedding semantics via multiheaded attention. Now we need to get
    #our final output

    x = self.norm(x) #final normalization layer, just normalize values in our vectors

    #output: logits and probabilities. Final head is just a single linear layer (we learn this too) mapping the all the vectors to logits
    #so we have an output for each token in x a vector of vocab size. This tells us for each word, how likely the next word is that at the corresponding
    #position for the logits in our vector.
    #We use logits directly for getting cross entropy loss in training, and we only care about the last logit vector position, aka
    #the logit vector for our current token, for predicting the next word and we just choose the highest value.

    logits = self.head(x)
    #probs = nn.Softmax(dim=-1)(logits) don't need to return these
    return logits


In [ ]:
def train(model, tokenizer, train_corpus, val_corpus, config):
    optimizer = torch.optim.Adam(model.parameters(), lr=config.lr)
    loss_fn   = nn.CrossEntropyLoss()
    model.to(device)

    # tokenize both corpora
    train_tokens = []
    for text in train_corpus:
        train_tokens += tokenizer.encode(text)

    val_tokens = []
    for text in val_corpus:
        val_tokens += tokenizer.encode(text)

    train_dataset = TextDataset(train_tokens, model.config.max_seq_len)
    val_dataset   = TextDataset(val_tokens,   model.config.max_seq_len)

    train_loader  = DataLoader(train_dataset, batch_size=config.batch_size, shuffle=True,  num_workers=4)
    val_loader    = DataLoader(val_dataset,   batch_size=config.batch_size, shuffle=False, num_workers=4)

    best_val_loss = float('inf')

    for epoch in range(config.n_epochs):
        # --- training pass ---
        model.train()
        total_train_loss = 0

        for input_ids, target_ids in train_loader:
            input_ids  = input_ids.to(device)
            target_ids = target_ids.to(device)

            optimizer.zero_grad()
            logits = model(input_ids)
            loss   = loss_fn(logits.view(-1, logits.shape[-1]), target_ids.view(-1))
            loss.backward()
            optimizer.step()
            total_train_loss += loss.item()

        avg_train_loss = total_train_loss / len(train_loader)

        # --- validation pass ---
        model.eval()
        total_val_loss = 0

        with torch.no_grad():
            for input_ids, target_ids in val_loader:
                input_ids  = input_ids.to(device)
                target_ids = target_ids.to(device)

                logits = model(input_ids)
                loss   = loss_fn(logits.view(-1, logits.shape[-1]), target_ids.view(-1))
                total_val_loss += loss.item()

        avg_val_loss = total_val_loss / len(val_loader)

        print(f"epoch {epoch:3d}  train loss {avg_train_loss:.4f}  val loss {avg_val_loss:.4f}")

        # save best checkpoint based on val loss, not train loss
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            torch.save(model.state_dict(), "toy_gpt_best.pt")
            print(f"  saved new best (val loss {avg_val_loss:.4f})")

In [ ]:
from transformers import GPT2Tokenizer

pretrained_tokenizer = GPT2Tokenizer.from_pretrained("gpt2")

In [ ]:
if "<|endoftext|>" not in tokenizer.vocab:
    tokenizer.vocab.append("<|endoftext|>")

# Create a new GPTConfig to ensure vocab_size is correctly updated
class DynamicGPTConfig(GPTConfig):
    vocab_size = 50257 #50257 for gpt2

model = ToyGPT(DynamicGPTConfig())
train(model, pretrained_tokenizer, train_corpus, val_corpus, TrainingConfig())

epoch   0  train loss 7.2821  val loss 6.6908
  saved new best (val loss 6.6908)
epoch   1  train loss 6.4168  val loss 6.3518
  saved new best (val loss 6.3518)
epoch   2  train loss 6.0367  val loss 6.1190
  saved new best (val loss 6.1190)
epoch   3  train loss 5.7437  val loss 5.9684
  saved new best (val loss 5.9684)
epoch   4  train loss 5.5092  val loss 5.8544
  saved new best (val loss 5.8544)
epoch   5  train loss 5.3095  val loss 5.7515
  saved new best (val loss 5.7515)
epoch   6  train loss 5.1334  val loss 5.6796
  saved new best (val loss 5.6796)
epoch   7  train loss 4.9741  val loss 5.6240
  saved new best (val loss 5.6240)
epoch   8  train loss 4.8275  val loss 5.5678
  saved new best (val loss 5.5678)
epoch   9  train loss 4.6910  val loss 5.5385
  saved new best (val loss 5.5385)
epoch  10  train loss 4.5601  val loss 5.5087
  saved new best (val loss 5.5087)
epoch  11  train loss 4.4345  val loss 5.4941
  saved new best (val loss 5.4941)
epoch  12  train loss 4.3132

KeyboardInterrupt: 

In [ ]:
def generate(model, tokenizer, prompt, max_new_tokens=20):
    model.eval()  # turn off dropout etc if you add them later

    # encode the prompt
    ids = tokenizer.encode(prompt)
    ids = torch.tensor(ids, dtype=torch.long).unsqueeze(0).to(device)  # (1, seq_len)

    with torch.no_grad():  # no gradients needed for inference
        for _ in range(max_new_tokens):
            # crop to max_seq_len if needed
            ids_cropped = ids[:, -model.config.max_seq_len:]

            # forward pass
            logits = model(ids_cropped)  # (1, seq_len, vocab_size)

            # take only the last position's logits — that's the next token prediction
            next_logits = logits[0, -1, :]  # (vocab_size,)

            # greedy — just pick the highest scoring token
            #next_id = torch.argmax(next_logits).item()
            #temperature sampling:
            temperature = 1.0
            probs = torch.softmax(next_logits / temperature, dim=-1)
            next_id = torch.multinomial(probs, num_samples=1).item()

            # append to sequence and loop
            ids = torch.cat([ids, torch.tensor([[next_id]], device=device)], dim=1)

    # decode the full sequence
    return tokenizer.decode(ids[0].tolist())


# test it
model.load_state_dict(torch.load("toy_gpt_best.pt"))
print(generate(model, pretrained_tokenizer, "The history of", max_new_tokens=50))
print(generate(model, pretrained_tokenizer, "In the United States", max_new_tokens=50))
print(generate(model, pretrained_tokenizer, "The game began", max_new_tokens=50))

The history of the game 's GameRankoring production . The idea was given to Lock Haven in 17 – 6 December 1908 . Helen experienced the record a set of regulating riff may be available in Texas . FA Cup multiplayer is also placed in the Cadets county line
In the United States by Sports Illustrated . The first is a piano place in the C Moderates lost piano . 
 On all 4 September 1939 , Florida Atlantic became the first Dutch system in Afghanistan to pave fire in Swansea for the BBC and fourth round in Australia with its
The game began to reach the Jarvis or gain out into the right . That 1 , the Activision must distance conditions across the M @-@ Dryden period , which had fluctchem freeway open points to the South East Omaha . Although its depleted fields were replaced by the


In [ ]:
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"The model has {total_params:,} trainable parameters.")

The model has 70,434,816 trainable parameters.
